# lore — VideoMAE + HuBERT with LoRA, 5-fold cross-validation on **Kaggle**

Same runs as `train_lore_colab.ipynb` (same code, configuration and folds, so the results are comparable). Only the
runtime changes: Kaggle gives **30 GPU-hours a week for free** and **two T4s**, so two folds train in parallel.

**Once, before the first run**

1. Kaggle account → Settings → **verify your phone number** (needed for GPU and Internet).
2. On your laptop: from this folder, `python scripts/pack_kaggle.py` → `kaggle_upload/lore-data.zip` (~650 MB)
   and `lore-code.zip`.
3. Kaggle → Datasets → New dataset, upload `lore-data.zip`, name it `lore-data`, **keep it Private**.
   Do the same with `lore-code.zip` (`lore-code`). **Never make them public**: the DOLOS terms forbid
   redistribution (`lie-detector/dataset/README.md` §12).
4. Upload this notebook (Create → Import notebook), then in the right panel:
   *Add Input* → your two datasets; *Session options* → Accelerator **GPU T4 x2**, Internet **On**.

**Running, in two versions**: *Save Version* → **Save & Run All (Commit)** runs in the background.

* **Version 1** trains **fold 1 only** (section 2) and stops: read its report in the log or in
  `runs/pilot-fold1/report.txt` (Output tab).
* **Version 2**, when fold 1 looks right: set `RUN_REMAINING_FOLDS = True` (section 3), attach version 1's
  Output as input, and commit again. Fold 1 is skipped; folds 2–5, fusion and the zip follow.

A version runs in the background (you can close the
browser) for at most 12 h. Results are written to `/kaggle/working/runs` and saved with the version's Output.

**Resuming** (if a version stopped before all folds were done, or to run the ablations later):
*Add Input* → *Notebook Output Files* → this notebook's previous version, then Save & Run All again.
Finished folds are copied and skipped; an interrupted fold resumes from its last epoch (`checkpoint.pt`).

**Code change?** Re-run `pack_kaggle.py --code-only` and upload `lore-code.zip` as a *new version* of `lore-code`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, time, torch
SESSION_START = time.time()
assert torch.cuda.is_available(), "No GPU: Session options -> Accelerator -> GPU T4 x2"
N_GPUS = torch.cuda.device_count()
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(torch.__version__, N_GPUS, 'x', torch.cuda.get_device_name(0), f"{GPU_GB:.0f} GB")

## 0 — Inputs, code, dependencies

In [ ]:
# Find the uploaded datasets wherever Kaggle mounted them (the /kaggle/input layout has changed over time)
import shutil, zipfile
from pathlib import Path

INPUT = Path('/kaggle/input')
FACE_CACHE = 'faces-yunet-t32-s224'
OWNER = 'Lorenzo'

def find_one(pattern, what):
    hits = sorted(INPUT.rglob(pattern), key=lambda p: len(p.parts))
    assert hits, f'{what} not found under {INPUT}: Add Input -> your private dataset (see the top cell)'
    return hits[0]

DATA = find_one('splits/manifest.json', 'dataset (dataset.csv + splits/)').parent.parent
assert (DATA / 'dataset.csv').exists() and (DATA / 'audio').is_dir(), f'{DATA} lacks dataset.csv or audio/'

# face cache: already extracted by Kaggle, or a zip to extract on the local disk
cache_cfg = [p for p in INPUT.rglob('cache_config.json') if p.parent.name == FACE_CACHE]
if cache_cfg:
    FACES_ROOT = cache_cfg[0].parent.parent
else:
    FACES_ROOT = Path('/tmp/faces')
    if not (FACES_ROOT / FACE_CACHE / 'cache_config.json').exists():
        with zipfile.ZipFile(find_one(f'{FACE_CACHE}.zip', 'face cache')) as z:
            z.extractall(FACES_ROOT)

# code: copied out of the read-only input so Python can write its caches
CODE_SRC = find_one('lore/train.py', 'code (lore-code dataset)').parent.parent
LORE = Path('/tmp/lore')
shutil.rmtree(LORE, ignore_errors=True)
shutil.copytree(CODE_SRC, LORE, ignore=shutil.ignore_patterns('runs', 'inputs', 'kaggle_upload', '__pycache__'))

RUNS = Path('/kaggle/working/runs')        # saved with the version's Output
RUNS.mkdir(parents=True, exist_ok=True)
print('data :', DATA)
print('faces:', FACES_ROOT / FACE_CACHE, len(list((FACES_ROOT / FACE_CACHE).glob('*.jpg'))), 'sprites')
print('code :', CODE_SRC, '->', LORE)

In [ ]:
# Resume: bring in the runs/ of any previous version attached as input
restored = []
for src_fold in sorted(p for p in INPUT.rglob('fold*') if p.is_dir() and p.parent.parent.name == 'runs'):
    run, fold = src_fold.parent.name, src_fold.name
    dst_fold = RUNS / run / fold
    if (dst_fold / 'metrics.json').exists():
        continue
    if (src_fold / 'metrics.json').exists() or (src_fold / 'checkpoint.pt').exists():
        shutil.copytree(src_fold, dst_fold, dirs_exist_ok=True)
        restored.append(f"{run}/{fold}: {'done' if (src_fold / 'metrics.json').exists() else 'checkpoint'}")
    if (src_fold.parent / 'run.json').exists() and not (RUNS / run / 'run.json').exists():
        shutil.copy2(src_fold.parent / 'run.json', RUNS / run / 'run.json')
for src_dir in INPUT.rglob('runs'):        # finished fusions, so they survive the next version
    for fusion in src_dir.glob('fusion-*'):
        if not (RUNS / fusion.name).exists():
            shutil.copytree(fusion, RUNS / fusion.name)
print('\n'.join(restored) or 'nothing to resume: fresh start')

In [ ]:
# The team repo, only for lie-detector/lib (splits + metrics, never copied)
if not os.path.isdir('/tmp/DeepLearningCourse/.git'):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /tmp/DeepLearningCourse
!git -C /tmp/DeepLearningCourse pull -q
# Pinned: the versions the code was tested with. transformers >= 5.6 is REQUIRED (older VideoMAE
# bypasses LoRA on query/value; the run would fail its LoRA check instead of training silently).
!pip install -q transformers==5.23.0 peft==0.21.3
# The Kaggle image ships torchao 0.10, and peft >= 0.18 raises ImportError when it finds a torchao older
# than 0.16 (even though LoRA never uses it). Nothing here needs torchao: remove it.
!pip uninstall -y -q torchao

In [ ]:
os.environ.update({
    'LORE_DATA': str(DATA),
    'LORE_LIB': '/tmp/DeepLearningCourse/lie-detector',
    'LORE_FACES_ROOT': str(FACES_ROOT),
    'LORE_RUNS': str(RUNS),
    # Kaggle kills a session at 12 h (and then may not save the Output): no epoch starts if it would
    # end after 11 h 15 min, so the version finishes cleanly and the next one resumes from checkpoint.pt
    'LORE_DEADLINE': str(SESSION_START + 11.25 * 3600),
    'PYTHONIOENCODING': 'utf-8',
    'HF_HUB_DISABLE_PROGRESS_BARS': '1',
    'TOKENIZERS_PARALLELISM': 'false',
})
# below 20 GB of GPU memory (T4, P100) the video branch recomputes activations instead of storing them
VIDEO_MEMORY_FLAGS = '--grad-checkpointing' if GPU_GB < 20 else ''
%cd {LORE}

In [ ]:
# Sanity checks (CPU, ~1 min): LoRA really in the forward pass; folds never touch the test set.
# A failure STOPS the notebook here (Kaggle marks the version as failed): no GPU time is wasted.
import subprocess

def run_or_stop(cmd, **env):
    proc = subprocess.Popen(cmd, shell=True, env=dict(os.environ, **env), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        if 'warn' not in line.lower():
            print(line, end='', flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f'failed (exit {proc.returncode}): {cmd} -- see the traceback above')

run_or_stop('python tests/test_lora_active.py')
run_or_stop('python tests/test_protocol.py')

In [ ]:
# Runs one command queue per GPU, in parallel, streaming the logs. A queue stops at its first
# failure; exit code 3 means "stopped at the deadline, resume in a new version" (not an error).
import threading

def _queue(gpu, commands, status):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    for cmd in commands:
        print(f'[gpu{gpu}] $ {cmd}', flush=True)
        proc = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            if 'warn' not in line.lower():
                print(f'[gpu{gpu}] {line}', end='', flush=True)
        code = proc.wait()
        status.append((gpu, cmd, code))
        if code != 0:
            print(f"[gpu{gpu}] {'STOPPED at the deadline' if code == 3 else f'FAILED (exit {code})'}: {cmd}",
                  flush=True)
            return

def run_on_gpus(queues, stagger_s=90):
    # stagger: the two processes must not download the same weights / write run.json at the same instant
    status, threads = [], []
    for gpu, commands in enumerate(queues):
        if not commands:
            continue
        threads.append(threading.Thread(target=_queue, args=(gpu, commands, status)))
        threads[-1].start()
        if gpu < len(queues) - 1:
            time.sleep(stagger_s)
    for t in threads:
        t.join()
    return status

def done(run, folds=range(1, 6)):
    return all((RUNS / run / f'fold{k}' / 'metrics.json').exists() for k in folds)

CV = 'python scripts/run_cv.py'

## 1 — Debug run (optional, ~10 min)

160/40/40 clips, 2 epochs, fold 1, written to local disk (not to `runs/`). It proves the loop works
on this GPU and prints the seconds per epoch and the peak memory: multiply by ~7 for a full epoch.
Skipped automatically once a real video fold exists.

In [ ]:
RUN_DEBUG = not (RUNS / 'video-lora-r8' / 'fold1' / 'metrics.json').exists() \
            and not (RUNS / 'video-lora-r8' / 'fold1' / 'checkpoint.pt').exists()
if RUN_DEBUG:
    run_or_stop(f'python scripts/run_cv.py --modality video --debug --folds 1 {VIDEO_MEMORY_FLAGS} '
                '--runs-dir /tmp/debug_runs', LORE_DEADLINE='')     # stops the notebook if it fails
else:
    print('debug run skipped: the video run has already started')

## 2 — Pilot: fold 1 only, then STOP and look at it

Trains **fold 1 only** for both branches (with two GPUs, audio and video at the same time), then prints:

* **sanity checks** (`lore/pilot.py`): the fold finished, the train loss went down, the predictions are not
  constant, the best val AUC is at least `MIN_VAL_AUC`; plus the real time of the fold and an estimate for folds 2–5;
* **performance of fold 1**, per branch: val and holdout ACC / F1 / AUC / balanced accuracy / precision /
  recall, the confusion matrix, holdout at the val-tuned threshold and at 0.5, the majority baseline, the
  learning curve (loss and val AUC per epoch) and accuracy by gender;
* **fold-1 late fusion** (video + audio, weight tuned on val), written to `runs/pilot-fold1/`.

Everything is also saved to `runs/pilot-fold1/report.txt` (Output tab). One fold is ~280 holdout clips:
read it as "does it learn and is it above the baseline", not as the final number.

The folds 2–5 cell below does **nothing** until you set `RUN_REMAINING_FOLDS = True` yourself.
If fold 1 is broken, fix the cause and re-run with a new run name (or delete `runs/<run>/fold1`):
a finished fold is never retrained.

In [ ]:
MIN_VAL_AUC = 0.5
RUNS_MAIN = ['audio-lora-r8', 'video-lora-r8']

import sys
sys.path.insert(0, str(LORE))
from lore.pilot import check_pilot, describe_fold

video = f'{CV} --modality video {VIDEO_MEMORY_FLAGS}'
audio = f'{CV} --modality audio'
if N_GPUS >= 2:
    run_on_gpus([[f'{audio} --folds 1'], [f'{video} --folds 1']])
else:
    run_on_gpus([[f'{audio} --folds 1', f'{video} --folds 1']])

PILOT_OK, lines = check_pilot(RUNS, RUNS_MAIN, fold=1, min_val_auc=MIN_VAL_AUC, remaining_folds=4, gpus=N_GPUS)
report = ['=== sanity checks ===', *lines, '', '=== fold 1 performance ===']
for run in RUNS_MAIN:
    report += [describe_fold(RUNS / run, 1), '']
PILOT_DIR = RUNS / 'pilot-fold1'
PILOT_DIR.mkdir(parents=True, exist_ok=True)
(PILOT_DIR / 'report.txt').write_text('\n'.join(report) + '\n', encoding='utf-8')
print('\n'.join(report))

In [ ]:
# fold-1 late fusion (only fold 1, written apart from the final fusion: it is not the board number)
if all((RUNS / r / 'fold1' / 'metrics.json').exists() for r in RUNS_MAIN):
    !python scripts/fuse.py --video-run video-lora-r8 --audio-run audio-lora-r8 --folds 1 --out-dir {PILOT_DIR} 2>&1 | grep -v -i warn | sed -n '1,8p'
else:
    print('fold-1 fusion skipped: a branch has no finished fold 1')

## 3 — Folds 2–5: started by hand

After reviewing fold 1: set `RUN_REMAINING_FOLDS = True` below, attach this version's Output as input
(*Add Input* → *Notebook Output Files*), and *Save & Run All* again. Fold 1 is then skipped (already done).

With two GPUs: GPU 0 runs audio folds 2–5 (~1 h) and then video folds 2 and 4; GPU 1 runs video folds 3
and 5. Same run folders as on Colab: `runs/audio-lora-r8/`, `runs/video-lora-r8/`. Finished folds are skipped.

In [ ]:
RUN_REMAINING_FOLDS = False     # set to True by hand once fold 1 looks right

if not RUN_REMAINING_FOLDS:
    print('folds 2-5 not started: review fold 1 above, then set RUN_REMAINING_FOLDS = True and run again')
else:
    if not PILOT_OK:
        print('WARNING: the fold-1 sanity checks failed (see above); starting folds 2-5 anyway, as requested')
    if N_GPUS >= 2:
        queues = [[f'{audio} --folds 2-5', f'{video} --folds 2,4'], [f'{video} --folds 3,5']]
    else:
        queues = [[f'{audio} --folds 2-5', f'{video} --folds 2-5']]
    status = run_on_gpus(queues)
print('\naudio complete:', done('audio-lora-r8'), '| video complete:', done('video-lora-r8'))

## 4 — Late fusion, report, results-board row

Only once both branches have all 5 folds. Re-running `run_cv.py` without `--folds` skips every fold and
rewrites `summary.json` over all five (the parallel processes each wrote a partial one).
Writes `runs/fusion-video-lora-r8+audio-lora-r8/` with `preds/cv_lore.csv` and `board_row.md`.

In [ ]:
if done('audio-lora-r8') and done('video-lora-r8'):
    !{CV} --modality audio 2>&1 | grep --line-buffered -v -i warn | tail -n 8
    !{CV} --modality video {VIDEO_MEMORY_FLAGS} 2>&1 | grep --line-buffered -v -i warn | tail -n 8
    !python scripts/fuse.py --video-run video-lora-r8 --audio-run audio-lora-r8 --owner "{OWNER}"
else:
    print('Not all folds are done yet: save this version, then start a new one with this version\'s '
          'Output attached as input (see the top cell). It resumes where this one stopped.')

## 5 — Ablations (optional, for the report)

* **head-only** — frozen backbones, only the linear head trained: is LoRA worth it? Cheap (no backward
  through the backbone). Runs in parallel on the two GPUs.
* **LoRA on the MLP of the last 4 blocks too** / **DoRA** — the variations listed in `docs/architectures.md`.

In [ ]:
RUN_ABLATIONS = False
if RUN_ABLATIONS:
    head_v = f'{CV} --modality video --head-only {VIDEO_MEMORY_FLAGS}'
    head_a = f'{CV} --modality audio --head-only'
    run_on_gpus([[head_a], [head_v]] if N_GPUS >= 2 else [[head_a, head_v]])
    if done('audio-headonly') and done('video-headonly'):
        !python scripts/fuse.py --video-run video-headonly --audio-run audio-headonly --owner "{OWNER}"
    # run_on_gpus([[f'{CV} --modality video --lora-mlp-last-n 4 {VIDEO_MEMORY_FLAGS}'],
    #              [f'{CV} --modality video --dora {VIDEO_MEMORY_FLAGS}']])

## 6 — The paper's official split (optional, reported separately — never on the board)

Same model and settings, trained and tested on the paper's **3 clip-level folds** (`official_fold{1,2,3}.json`),
to compare with the paper's numbers. Run names end in `-official`; they never mix with the subject-level runs.

* The paper's folds have no validation set, so a fixed, stratified **15% of each fold's train** is set aside for
  early stopping and the threshold (same clips for audio and video). The test fold is untouched; the 0–1 clips
  the paper lists on both sides stay in test only.
* Two numbers per branch: **ours** (epoch and threshold chosen on that val, the test fold scored once) and
  **paper-style** (the paper's `train_test.py` keeps the epoch with the best accuracy *on the test fold* at
  threshold 0.5 — optimistic). Only the paper-style one is comparable with the paper's 64.75%.
* Why it is not on the board: the folds are clip-level, so the same people (and episodes) are in train and test,
  including people of the reserved test set. It measures something easier than our protocol.

With two GPUs: GPU 0 runs audio folds 1–3 and video fold 1, GPU 1 video folds 2–3 (about the cost of 3
subject-level folds). Results: `runs/fusion-video-lora-r8-official+audio-lora-r8-official/comparison.txt`.

In [ ]:
RUN_OFFICIAL = False    # set to True by hand to run the paper-split comparison
if RUN_OFFICIAL:
    off_v = f'{CV} --modality video --protocol official {VIDEO_MEMORY_FLAGS}'
    off_a = f'{CV} --modality audio --protocol official'
    if N_GPUS >= 2:
        run_on_gpus([[off_a, f'{off_v} --folds 1'], [f'{off_v} --folds 2,3']])
    else:
        run_on_gpus([[off_a, off_v]])
    if done('audio-lora-r8-official', range(1, 4)) and done('video-lora-r8-official', range(1, 4)):
        !{CV} --modality video --protocol official {VIDEO_MEMORY_FLAGS} 2>&1 | grep -v -i warn | tail -n 5
        !{CV} --modality audio --protocol official 2>&1 | grep -v -i warn | tail -n 5
        !python scripts/fuse.py --video-run video-lora-r8-official --audio-run audio-lora-r8-official --owner "{OWNER}"
    else:
        print('official split: not all 3 folds are done; run a new version with this Output attached')

## 7 — One zip to download (Output tab → `lore_runs.zip`)

In [ ]:
shutil.make_archive('/kaggle/working/lore_runs', 'zip', RUNS.parent, RUNS.name)
print(f"{Path('/kaggle/working/lore_runs.zip').stat().st_size / 1e6:.1f} MB, "
      f"session {(time.time() - SESSION_START) / 3600:.1f} h")
!find {RUNS} -name metrics.json | sort